In [1]:
import pandas as pd
import numpy as np

df_freq = pd.read_csv("../../data/freMTPL2freq.csv")
df_sev = pd.read_csv("../../data/df_sev_clean.csv")

# ۱. فیلتر کردن ادعاهایی که مبلغ خسارت آن‌ها بزرگتر از صفر است
claims_data = df_sev[df_sev['ClaimAmount'] > 0].copy()

# ۲. مرتب‌سازی ادعاها به صورت نزولی بر اساس مبلغ خسارت
claims_data = claims_data.sort_values(by='ClaimAmount', ascending=False).reset_index(drop=True)

# ۳. محاسبه سهم تجمعی هزینه و سهم تجمعی تعداد ادعاها
claims_data['Cumulative_Cost'] = claims_data['ClaimAmount'].cumsum()
claims_data['Cumulative_Cost_Pct'] = claims_data['Cumulative_Cost'] / claims_data['ClaimAmount'].sum()
claims_data['Cumulative_Claim_Pct'] = (claims_data.index + 1) / len(claims_data)

# ۴. محاسبه سهم ۱٪، ۵٪ و ۱۰٪ پرهزینه‌ترین ادعاها از کل خسارت
top_1_pct_cost = claims_data[claims_data['Cumulative_Claim_Pct'] <= 0.01]['Cumulative_Cost_Pct'].max()
top_5_pct_cost = claims_data[claims_data['Cumulative_Claim_Pct'] <= 0.05]['Cumulative_Cost_Pct'].max()
top_10_pct_cost = claims_data[claims_data['Cumulative_Claim_Pct'] <= 0.10]['Cumulative_Cost_Pct'].max()

pareto_summary = pd.DataFrame({
    'Top % of Claims': ['Top 1%', 'Top 5%', 'Top 10%'],
    'Share of Total Claim Cost': [
        f"{top_1_pct_cost * 100:.2f}%",
        f"{top_5_pct_cost * 100:.2f}%",
        f"{top_10_pct_cost * 100:.2f}%"
    ]
})

print(pareto_summary)

# تنها ۱٪ از پرهزینه‌ترین پرونده‌های خسارت،
# حدود ۳۸٪ از کل هزینه خسارت را تشکیل می‌دهند.
# این نتیجه نشان می‌دهد هزینه‌ها در تعداد کمی از ادعاها متمرکز شده‌اند.

  Top % of Claims Share of Total Claim Cost
0          Top 1%                    37.99%
1          Top 5%                    52.09%
2         Top 10%                    59.92%
